# RUNG 13 + 14 — death-wave + the MECHANISM ARENA (CPU, no GPU)

**Run it all at once.** This notebook runs two things and tells you which cancer-killing strategy *hits*:

1. **RUNG-13** (`scripts/38`) — the validated single-cell EARM death switch wired into a coupled **death wave**: all-or-none threshold, variable delay, snap, irreversible commitment, and a *bounded* wave that confirms the RUNG-12P/B percolation under real kinetics.
2. **RUNG-14 — the arena** (`scripts/39`) — **7 rigorous strategies + 2 toy physics arms** across **many regimes** (time horizon, dose, recognition-leak q_n). Output = a **leaderboard**: which strategy clears tumour *and* spares normal across regimes (HIT), which only works in a window (CLOSE), which doesn't contain itself (FAR).

Strategies: `per_cell` (baseline, linear leak) · `wave` (bystander) · `quorum` (density-gated) · `diffusible` (GDEPT) · `oncolytic` (self-amplifying) · `alt_death` (reroute apoptosis-resistant cells) · `combo` (wave + BH3-mimetic). Toy/future: `oncotripsy` (sound), `ttfields` (EM).

**Honest ceiling:** every arm's recognition→effector coupling is a PROXY parameter; mapping it to a real molecular/delivery efficiency is the wet-lab residual. Toy physics arms are cartoons for triage. CPU-only, **Run all**, ~a few minutes.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log (numpy-only; no GPU/atlas needed)
from scripts.runlog import new_log
RUNLOG = new_log('rung13_14_arena', repo_dir='.')
print('[CELL 2] ✓ logging to', RUNLOG)

In [ ]:
#@title Cell 3 — VALIDATE both ports (fast, synthetic). Stop if anything fails.
import sys
from scripts.runlog import run_logged
rc1 = run_logged([sys.executable,'-u','scripts/38_apoptosis_wave.py','selftest'], RUNLOG)
rc2 = run_logged([sys.executable,'-u','scripts/39_mechanism_arena.py','selftest'], RUNLOG)
print('[CELL 3]', '✓ both validated' if (rc1==0 and rc2==0) else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — RUNG-13: single-cell switch → coupled death wave (real kinetics)
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/38_apoptosis_wave.py','run'], RUNLOG)
print('[CELL 4]', '✓ done' if rc==0 else f'✗ exit {rc}')
from IPython.display import Image, display
if os.path.exists('runs/rung13_apoptosis/rung13_apoptosis.png'): display(Image('runs/rung13_apoptosis/rung13_apoptosis.png'))
if os.path.exists('runs/rung13_apoptosis/rung13_apoptosis.json'):
    d=json.load(open('runs/rung13_apoptosis/rung13_apoptosis.json')); print('DECISIVE:', d['DECISIVE'])

In [ ]:
#@title Cell 5 — RUNG-14: THE ARENA (many strategies × many regimes). Watch the LEADERBOARD.
#@markdown ~a few minutes on CPU. Set `quick` for a fast first look, or full for all regimes.
mode = 'run' #@param ['run', 'quick']
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/39_mechanism_arena.py', mode], RUNLOG)
print('[CELL 5]', '✓ done' if rc==0 else f'✗ exit {rc}')
from IPython.display import Image, display
if os.path.exists('runs/rung14_arena/rung14_arena.png'): display(Image('runs/rung14_arena/rung14_arena.png'))
if os.path.exists('runs/rung14_arena/rung14_arena.json'):
    d=json.load(open('runs/rung14_arena/rung14_arena.json'))
    print('LEADERBOARD:', ' > '.join(f"{b['mechanism']}({b['safe_fraction']:.2f})" for b in d['leaderboard_rigorous']))
    print('\nDECISIVE:', d['DECISIVE'])

In [ ]:
#@title Cell 6 — bundle ONE zip + download (figs + JSON + log)
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung13_14_arena_','').replace('.log','')
b = f'/content/rung13_14_run_{stem}.zip'
ps = sorted(set(glob.glob('runs/rung13_apoptosis/*.png')+glob.glob('runs/rung13_apoptosis/*.json')
                 +glob.glob('runs/rung14_arena/*.png')+glob.glob('runs/rung14_arena/*.json')+[str(RUNLOG)]))
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')